# Five-threshold directional-change linkage

This notebook links four adjacent causal building blocks for the frozen GBPUSD.r constant-ratio hierarchy:

1. small 0.20% → medium 0.30%;
2. medium 0.30% → large 0.45%;
3. large 0.45% → macro 0.675%;
4. macro 0.675% → outer 1.0125%.

A valid lineage begins when all five modes agreed on the previous bar and a small DCC establishes a counter-scale target. Exact event IDs—not nearest timestamps—carry the lineage through later hand-offs.

Terminal depth is a hindsight description:

- 0: rejected before medium;
- 1: reached medium, rejected before large;
- 2: reached large, rejected before macro;
- 3: reached macro, rejected before outer;
- 4: reached outer.

The constant-ratio grid was selected for equal adjacent geometry, not because of its realized propagation outcomes.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
CASEBOOK_BARS_BEFORE = 180
CASEBOOK_BARS_AFTER = 180

In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"
    OUTER = "outer"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"
    LARGE_TO_MACRO = "large_to_macro"
    MACRO_TO_OUTER = "macro_to_outer"


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    STOPPED_BELOW_MACRO = "stopped_below_macro"
    STOPPED_BELOW_OUTER = "stopped_below_outer"
    REACHED_OUTER = "reached_outer"
    AMBIGUOUS_STAGE = "ambiguous_stage"
    UNRESOLVED_STAGE = "unresolved_stage"
    UNLINKED_HANDOFF = "unlinked_handoff"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO, Scale.OUTER)
THRESHOLDS = {
    Scale.SMALL: 0.002,
    Scale.MEDIUM: 0.003,
    Scale.LARGE: 0.0045,
    Scale.MACRO: 0.00675,
    Scale.OUTER: 0.010125,
}
SCALE_PREFIXES = {
    Scale.SMALL: "S",
    Scale.MEDIUM: "M",
    Scale.LARGE: "L",
    Scale.MACRO: "X",
    Scale.OUTER: "O",
}
SCALE_COLORS = {
    Scale.SMALL: "#16a34a",
    Scale.MEDIUM: "#2563eb",
    Scale.LARGE: "#7c3aed",
    Scale.MACRO: "#c026d3",
    Scale.OUTER: "#0891b2",
}
STAGE_ORDER = (
    Stage.SMALL_TO_MEDIUM,
    Stage.MEDIUM_TO_LARGE,
    Stage.LARGE_TO_MACRO,
    Stage.MACRO_TO_OUTER,
)
STAGE_SPECS = {
    Stage.SMALL_TO_MEDIUM: (Scale.SMALL, Scale.MEDIUM),
    Stage.MEDIUM_TO_LARGE: (Scale.MEDIUM, Scale.LARGE),
    Stage.LARGE_TO_MACRO: (Scale.LARGE, Scale.MACRO),
    Stage.MACRO_TO_OUTER: (Scale.MACRO, Scale.OUTER),
}
STOPPED_OUTCOMES = {
    0: TerminalOutcome.STOPPED_BELOW_MEDIUM,
    1: TerminalOutcome.STOPPED_BELOW_LARGE,
    2: TerminalOutcome.STOPPED_BELOW_MACRO,
    3: TerminalOutcome.STOPPED_BELOW_OUTER,
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

threshold_table = pd.DataFrame({
    "threshold": [THRESHOLDS[scale] for scale in SCALE_ORDER],
    "ratio_to_previous": [
        np.nan,
        *[
            THRESHOLDS[upper] / THRESHOLDS[lower]
            for lower, upper in zip(SCALE_ORDER[:-1], SCALE_ORDER[1:], strict=True)
        ],
    ],
}, index=[scale.value for scale in SCALE_ORDER])
threshold_table.index.name = "scale"
threshold_table.style.format({"threshold": "{:.4%}", "ratio_to_previous": "{:.3f}"})

,threshold,ratio_to_previous
scale,,
small,0.2000%,nan
medium,0.3000%,1.500
large,0.4500%,1.500
macro,0.6750%,1.500
outer,1.0125%,1.500


## Genealogical and causal contract

The previous bar must have all five modes aligned. The initiating small DCC establishes the immutable lineage ID and target direction.

A stage propagates only through the exact upper event that resolved the previous stage. That event must be the initiating lower event of the next adjacent contest. Rejection, ambiguity, an unresolved sample end, or a missing exact hand-off terminates the lineage.

At initiation only the seed, target mode, previous five-mode agreement and completed small leg are known. Terminal depth and all stage outcomes remain hindsight until their resolution DCCs.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Symbol: {SYMBOL}")
print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Symbol: GBPUSD.r
Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger as _prepare_event_ledger

def prepare_event_ledger(source: pd.DataFrame, scale: Scale, theta: float) -> pd.DataFrame:
    return _prepare_event_ledger(source, SCALE_PREFIXES[scale], scale, theta)


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    theta = THRESHOLDS[scale]
    ledger = prepare_event_ledger(parse_dc_events(prices=prices, theta=theta), scale, theta)
    live = dc_live_features(prices, theta=theta)
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    event_lookup = ledger.set_index("dcc_idx")["event_id"]
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)

clock_summary = pd.DataFrame.from_records([
    {
        "scale": scale.value,
        "threshold": THRESHOLDS[scale],
        "events": len(event_ledgers[scale]),
        "median_t": event_ledgers[scale]["t"].median(),
        "median_tmv": event_ledgers[scale]["tmv"].median(),
        "median_r": event_ledgers[scale]["r"].median(),
    }
    for scale in SCALE_ORDER
]).set_index("scale")
clock_summary.style.format({
    "threshold": "{:.4%}",
    "events": "{:,}",
    "median_t": "{:.1f}",
    "median_tmv": "{:.3f}",
    "median_r": "{:.3e}",
})

,threshold,events,median_t,median_tmv,median_r
scale,,,,,
small,0.2000%,"2,697",60.0,1.858,6.655e-05
medium,0.3000%,"1,389",136.0,1.771,4.471e-05
large,0.4500%,683,275.0,1.693,3.007e-05
macro,0.6750%,308,581.0,1.753,2.142e-05
outer,1.0125%,146,1421.5,1.680,1.430e-05


## Reconstruct the four adjacent contests

Each adjacent contest is built independently from causal DCC flags. Same-bar resolutions are retained as ambiguous timing cases for audit but excluded from the strict-clean lineage cohort.

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


attempts_by_stage = {}
for stage in STAGE_ORDER:
    lower_scale, upper_scale = STAGE_SPECS[stage]
    attempts = build_adjacent_attempts(mode_frame, lower_scale, upper_scale)
    attempts.insert(0, "stage", stage.value)
    attempts_by_stage[stage] = attempts

adjacent_summary = pd.DataFrame.from_records([
    {
        "stage": stage.value,
        "attempts": len(attempts_by_stage[stage]),
        "propagated": attempts_by_stage[stage]["outcome"].eq(AttemptOutcome.PROPAGATED.value).sum(),
        "rejected": attempts_by_stage[stage]["outcome"].eq(AttemptOutcome.REJECTED.value).sum(),
        "ambiguous": attempts_by_stage[stage]["outcome"].eq(AttemptOutcome.AMBIGUOUS.value).sum(),
        "unresolved": attempts_by_stage[stage]["outcome"].eq(AttemptOutcome.UNRESOLVED.value).sum(),
    }
    for stage in STAGE_ORDER
]).set_index("stage")
adjacent_summary

,attempts,propagated,rejected,ambiguous,unresolved
stage,,,,,
small_to_medium,2043,1389,654,0,0
medium_to_large,1036,683,353,0,0
large_to_macro,496,308,187,0,1
macro_to_outer,226,146,80,0,0


## Pin broad-agreement small seeds

A root is eligible only when the previous bar had all five modes equal. This gives the small counter-DCC a definite common-scale ancestry. Eligibility uses no future outcome.

In [6]:
mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
previous_modes = mode_frame[mode_columns].shift(1)
small_attempts = attempts_by_stage[Stage.SMALL_TO_MEDIUM].copy()
seed_times = pd.DatetimeIndex(small_attempts["start_time"])
seed_previous_modes = previous_modes.loc[seed_times]
all_previous_known = seed_previous_modes.notna().all(axis=1)
broad_previous_agreement = seed_previous_modes.nunique(axis=1).eq(1)
eligible_seed_mask = (all_previous_known & broad_previous_agreement).to_numpy()
seeds = small_attempts.loc[eligible_seed_mask].copy()

if seeds["initiating_event_id"].duplicated().any():
    raise AssertionError("Eligible seed event IDs must be unique.")

seed_summary = pd.Series({
    "all_small_to_medium_attempts": len(small_attempts),
    "broad_agreement_seeds": len(seeds),
    "seed_share": len(seeds) / len(small_attempts),
}, name="value")
seed_summary.to_frame()

,value
all_small_to_medium_attempts,2043.000000
broad_agreement_seeds,799.000000
seed_share,0.391092


## Link all four stages by exact event ID

The long stage table retains every traversed stage. The wide lineage table records terminal depth and status. A clean lineage must resolve without ambiguity, must have every required exact hand-off, and must contain at least one pre-resolution bar in every traversed stage.

In [7]:
from dc_basics.lineage import build_five_scale_linkages


linkages, linkage_stages = build_five_scale_linkages(seeds, attempts_by_stage)
strict_clean_linkages = linkages.loc[linkages["strict_clean"]].copy()
strict_lineage_ids = set(strict_clean_linkages["lineage_id"])
strict_clean_stages = linkage_stages.loc[linkage_stages["lineage_id"].isin(strict_lineage_ids)].copy()

if strict_clean_linkages["lineage_id"].duplicated().any():
    raise AssertionError("Strict-clean lineage IDs must be unique.")
if not strict_clean_linkages["terminal_depth"].between(0, 4).all():
    raise AssertionError("Strict-clean terminal depth must be between zero and four.")
if not strict_clean_stages["resolution_bars"].gt(0).all():
    raise AssertionError("A strict-clean stage has no pre-resolution bars.")

status_summary = linkages.groupby("terminal_outcome", dropna=False).agg(
    lineages=("lineage_id", "size"),
    same_bar_stages=("same_bar_stage_count", "sum"),
)
status_summary["share"] = status_summary["lineages"] / len(linkages)
display(status_summary.style.format({"lineages": "{:,}", "same_bar_stages": "{:,}", "share": "{:.2%}"}))
print(f"All eligible seeds: {len(linkages):,}")
print(f"Strict-clean lineages: {len(strict_clean_linkages):,}")

,lineages,same_bar_stages,share
terminal_outcome,,,
reached_outer,146,34,18.27%
stopped_below_large,180,8,22.53%
stopped_below_macro,130,10,16.27%
stopped_below_medium,262,0,32.79%
stopped_below_outer,80,16,10.01%
unresolved_stage,1,0,0.13%


All eligible seeds: 799
Strict-clean lineages: 751


## Exact hand-off integrity

For every propagated non-terminal stage, the resolution event ID must equal the next stage's initiating event ID. Stage starts and resolutions must be chronological, and the number of traversed stages must equal terminal depth plus one except for depth four, which traverses all four stages.

In [8]:
strict_stage_order = strict_clean_stages.sort_values(["lineage_id", "stage_order"])
for lineage in strict_clean_linkages.itertuples(index=False):
    stages = strict_stage_order.loc[strict_stage_order["lineage_id"].eq(lineage.lineage_id)]
    expected_stage_count = 4 if int(lineage.terminal_depth) == 4 else int(lineage.terminal_depth) + 1
    if len(stages) != expected_stage_count:
        raise AssertionError("Traversed-stage count disagrees with terminal depth.")
    if not stages["start_position"].is_monotonic_increasing:
        raise AssertionError("Linked stage starts are not chronological.")
    if not stages["resolution_position"].astype(int).is_monotonic_increasing:
        raise AssertionError("Linked stage resolutions are not chronological.")
    for current, following in zip(stages.iloc[:-1].itertuples(), stages.iloc[1:].itertuples(), strict=True):
        if current.outcome != AttemptOutcome.PROPAGATED.value:
            raise AssertionError("A non-propagated stage was followed by another stage.")
        if current.resolution_event_id != following.initiating_event_id:
            raise AssertionError("An exact event-ID hand-off failed.")
        if int(current.resolution_position) != int(following.start_position):
            raise AssertionError("A hand-off event changed bar position.")

integrity_summary = strict_clean_stages.groupby("stage", sort=False).agg(
    traversed_stages=("lineage_id", "size"),
    unique_lineages=("lineage_id", "nunique"),
    median_resolution_bars=("resolution_bars", "median"),
)
integrity_summary

,traversed_stages,unique_lineages,median_resolution_bars
stage,,,
small_to_medium,751,751,18.0
medium_to_large,489,489,43.0
large_to_macro,317,317,102.0
macro_to_outer,196,196,234.5


## Linked probability ladder

Every probability uses the same strict-clean seed cohort. Conditional rates describe each stage among lineages that reached it. Cumulative reach is arithmetic conditioning, not an independence assumption.

In [9]:
reach_labels = ["seed", "medium", "large", "macro", "outer"]
probability_records = []
cohort_size = len(strict_clean_linkages)

for depth, label in enumerate(reach_labels):
    if depth == 0:
        reached = cohort_size
        conditional_rate = 1.0
    else:
        reached = strict_clean_linkages["terminal_depth"].ge(depth).sum()
        previous_reached = strict_clean_linkages["terminal_depth"].ge(depth - 1).sum()
        conditional_rate = reached / previous_reached
    probability_records.append({
        "attained_scale": label,
        "minimum_depth": depth,
        "lineages_reaching_scale": reached,
        "conditional_reach_rate": conditional_rate,
        "cumulative_reach_rate": reached / cohort_size,
    })

probability_ladder = pd.DataFrame.from_records(probability_records).set_index("attained_scale")
terminal_depth_summary = strict_clean_linkages["terminal_depth"].astype(int).value_counts().sort_index()
terminal_depth_summary = terminal_depth_summary.reindex(range(5), fill_value=0).rename("lineages").to_frame()
terminal_depth_summary["share"] = terminal_depth_summary["lineages"] / cohort_size

display(probability_ladder.style.format({
    "minimum_depth": "{:,}",
    "lineages_reaching_scale": "{:,}",
    "conditional_reach_rate": "{:.2%}",
    "cumulative_reach_rate": "{:.2%}",
}))
display(terminal_depth_summary.style.format({"lineages": "{:,}", "share": "{:.2%}"}))

depth_figure = make_subplots(rows=1, cols=2, subplot_titles=["Terminal depth", "Cumulative scale reach"])
depth_figure.add_trace(go.Bar(
    x=terminal_depth_summary.index,
    y=terminal_depth_summary["lineages"],
    marker_color="#7c3aed",
    name="Terminal depth",
), row=1, col=1)
depth_figure.add_trace(go.Bar(
    x=probability_ladder.index,
    y=probability_ladder["cumulative_reach_rate"],
    marker_color="#0891b2",
    name="Cumulative reach",
), row=1, col=2)
depth_figure.update_layout(
    title=f"{SYMBOL}: strict-clean five-scale linkage flow",
    template="plotly_white",
    height=500,
    showlegend=False,
)
depth_figure.update_yaxes(title="Lineages", row=1, col=1)
depth_figure.update_yaxes(title="Cumulative rate", tickformat=".0%", row=1, col=2)
depth_figure.show()

,minimum_depth,lineages_reaching_scale,conditional_reach_rate,cumulative_reach_rate
attained_scale,,,,
seed,0,751,100.00%,100.00%
medium,1,489,65.11%,65.11%
large,2,317,64.83%,42.21%
macro,3,196,61.83%,26.10%
outer,4,126,64.29%,16.78%


,lineages,share
terminal_depth,,
0,262,34.89%
1,172,22.90%
2,121,16.11%
3,70,9.32%
4,126,16.78%


## Stage resolution anatomy

These summaries describe resolved paths with hindsight. Duration and intervening event counts are not initiation features. They show how much lower-scale activity occurred before each stage resolved.

In [10]:
from dc_basics.audit import count_events_inside


event_positions = {
    scale: event_ledgers[scale]["dcc_pos"].to_numpy(dtype=np.int64)
    for scale in SCALE_ORDER
}
anatomy_records = []
for stage_row in strict_clean_stages.itertuples(index=False):
    record = {
        "lineage_id": stage_row.lineage_id,
        "stage": stage_row.stage,
        "outcome": stage_row.outcome,
        "target_mode": stage_row.target_mode,
        "resolution_bars": int(stage_row.resolution_bars),
    }
    for scale in SCALE_ORDER:
        record[f"{scale.value}_dccs_during_stage"] = count_events_inside(
            event_positions[scale],
            int(stage_row.start_position),
            int(stage_row.resolution_position),
        )
    anatomy_records.append(record)

stage_anatomy = pd.DataFrame.from_records(anatomy_records)
stage_summary = stage_anatomy.groupby(["stage", "outcome"], sort=False).agg(
    stages=("lineage_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
    p90_resolution_bars=("resolution_bars", lambda values: values.quantile(0.90)),
    median_small_dccs=("small_dccs_during_stage", "median"),
    median_medium_dccs=("medium_dccs_during_stage", "median"),
    median_large_dccs=("large_dccs_during_stage", "median"),
    median_macro_dccs=("macro_dccs_during_stage", "median"),
    median_outer_dccs=("outer_dccs_during_stage", "median"),
)
display(stage_summary.style.format({
    "stages": "{:,}",
    "median_resolution_bars": "{:.1f}",
    "p90_resolution_bars": "{:.1f}",
    "median_small_dccs": "{:.1f}",
    "median_medium_dccs": "{:.1f}",
    "median_large_dccs": "{:.1f}",
    "median_macro_dccs": "{:.1f}",
    "median_outer_dccs": "{:.1f}",
}))

## Directional and chronological stability

Terminal depth remains hindsight. These strata test whether the linked flow is concentrated in one direction or one contiguous part of the sample. They are descriptive robustness checks, not trading conditions.

In [11]:
strict_clean_linkages["sample_third"] = pd.cut(
    strict_clean_linkages["seed_position"] / (len(prices) - 1),
    bins=[0.0, 1.0 / 3.0, 2.0 / 3.0, 1.0],
    labels=["early", "middle", "late"],
    include_lowest=True,
)
direction_depth = pd.crosstab(
    strict_clean_linkages["target_mode"],
    strict_clean_linkages["terminal_depth"].astype(int),
    normalize="index",
)
period_depth = pd.crosstab(
    strict_clean_linkages["sample_third"],
    strict_clean_linkages["terminal_depth"].astype(int),
    normalize="index",
).reindex(["early", "middle", "late"], fill_value=0.0)
display(direction_depth.style.format("{:.2%}"))
display(period_depth.style.format("{:.2%}"))

terminal_depth,0,1,2,3,4
target_mode,,,,,
down,36.41%,22.05%,15.90%,8.97%,16.67%
up,33.24%,23.82%,16.34%,9.70%,16.90%


terminal_depth,0,1,2,3,4
sample_third,,,,,
early,32.79%,23.36%,16.39%,11.07%,16.39%
middle,34.58%,23.39%,17.29%,7.80%,16.95%
late,37.74%,21.70%,14.15%,9.43%,16.98%


## Five-depth price-path casebook

One recent strict-clean lineage is selected for each observed terminal depth. Selection and terminal markers are hindsight. Stage bands show the exact linked intervals; all DCC markers were causally observed on their bars.

In [12]:
casebook_lineages = (
    strict_clean_linkages.sort_values("seed_time")
    .groupby("terminal_depth", sort=True, group_keys=False)
    .tail(1)
    .sort_values("terminal_depth")
)
display(casebook_lineages[[
    "lineage_id", "target_mode", "seed_time", "terminal_time",
    "terminal_depth", "terminal_outcome",
]])

stage_color_map = {
    Stage.SMALL_TO_MEDIUM.value: "#22c55e",
    Stage.MEDIUM_TO_LARGE.value: "#3b82f6",
    Stage.LARGE_TO_MACRO.value: "#8b5cf6",
    Stage.MACRO_TO_OUTER.value: "#ec4899",
}

,lineage_id,target_mode,seed_time,terminal_time,terminal_depth,terminal_outcome
797,S2695,up,2026-11-06 17:20:00+02:00,2026-11-06 21:45:00+02:00,0,stopped_below_medium
795,S2690,down,2026-11-04 17:00:00+02:00,2026-11-05 06:10:00+02:00,1,stopped_below_large
787,S2660,down,2026-10-16 17:25:00+02:00,2026-10-20 08:25:00+02:00,2,stopped_below_macro
785,S2649,up,2026-10-08 14:25:00+02:00,2026-10-14 15:50:00+02:00,3,stopped_below_outer
796,S2692,down,2026-11-05 09:30:00+02:00,2026-11-06 15:25:00+02:00,4,reached_outer


In [13]:
def plot_linkage_casebook(
    close: pd.Series, examples: pd.DataFrame, stages: pd.DataFrame
) -> go.Figure:
    figure = make_subplots(
        rows=len(examples),
        cols=1,
        vertical_spacing=min(0.04, 0.15 / len(examples)),
        subplot_titles=[
            f"depth={int(example.terminal_depth)} | {example.terminal_outcome} | {example.lineage_id}"
            for example in examples.itertuples(index=False)
        ],
    )

    for row_number, example in enumerate(examples.itertuples(index=False), start=1):
        window_start = max(0, int(example.seed_position) - CASEBOOK_BARS_BEFORE)
        window_end = min(len(close), int(example.terminal_position) + CASEBOOK_BARS_AFTER + 1)
        price_window = close.iloc[window_start:window_end]
        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#64748b", "width": 1.1},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)

        lineage_stages = stages.loc[stages["lineage_id"].eq(example.lineage_id)].sort_values("stage_order")
        for stage_row in lineage_stages.itertuples(index=False):
            figure.add_vrect(
                x0=stage_row.start_time,
                x1=stage_row.resolution_time,
                fillcolor=stage_color_map[stage_row.stage],
                opacity=0.16,
                line_width=0,
                row=row_number,
                col=1,
            )
            figure.add_vline(
                x=stage_row.resolution_time,
                line={"color": stage_color_map[stage_row.stage], "width": 1.2, "dash": "dot"},
                row=row_number,
                col=1,
            )

        for scale in SCALE_ORDER:
            ledger = event_ledgers[scale]
            visible = ledger.loc[ledger["dcc_idx"].between(price_window.index[0], price_window.index[-1])]
            if visible.empty:
                continue
            figure.add_trace(go.Scatter(
                x=visible["dcc_idx"],
                y=visible["dcc_price"],
                mode="markers",
                name=f"{scale.value.title()} DCC",
                legendgroup=f"{scale.value}_dcc",
                showlegend=row_number == 1,
                marker={
                    "color": SCALE_COLORS[scale],
                    "size": 4 + SCALE_ORDER.index(scale),
                    "symbol": "x",
                },
                customdata=visible[["event_id", "direction"]].to_numpy(),
                hovertemplate=(
                    f"{scale.value.title()} DCC<br>time=%{{x}}<br>price=%{{y:.5f}}<br>"
                    "event=%{customdata[0]}<br>completed direction=%{customdata[1]}<extra></extra>"
                ),
            ), row=row_number, col=1)

        terminal_price = close.iloc[int(example.terminal_position)]
        figure.add_trace(go.Scatter(
            x=[example.terminal_time],
            y=[terminal_price],
            mode="markers",
            name="Terminal resolution",
            legendgroup="terminal",
            showlegend=row_number == 1,
            marker={"color": "#111827", "size": 10, "symbol": "diamond"},
            hovertemplate="terminal resolution<br>time=%{x}<br>price=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: five-scale exact-linkage casebook",
        template="plotly_white",
        height=max(1000, 310 * len(examples)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.04, "x": 0.0},
    )
    figure.update_xaxes(title="Time", row=len(examples), col=1)
    return figure


plot_linkage_casebook(prices, casebook_lineages, strict_clean_stages).show()

# Five-threshold linkage completion gate

Before moving to causal live development, be able to explain:

1. Why a valid root requires broad agreement on the previous bar.
2. Why the small seed remains the immutable lineage ID through four possible hand-offs.
3. Why each next stage must be found by exact resolution event ID.
4. Why nearest-time matching would create false genealogy.
5. Why terminal depth is hindsight until the resolving DCC.
6. Why attained depth is causal once a hand-off has occurred.
7. Why a depth-zero lineage still contains a real small counter-scale attempt.
8. Why depth four means the target reached outer, not that the move must continue.
9. Why same-bar stages are excluded from the strict-clean cohort.
10. Why conditional reach rates multiply arithmetically without assuming stage independence.
11. How much lower-scale activity can occur while a later stage remains unresolved.
12. Whether depth distributions are reasonably stable by direction and sample third.
13. Why the linkage is structural description rather than a trading signal.
14. Why the next notebook must exclude each resolution bar from its live feature panel.

The next step is five-threshold causal live development: retain the seed, update only information known on each pre-resolution bar, and audit how the hierarchy develops while terminal depth is still unknown.